# EDA Visualization Storytelling

**Goal:** produce a short analytical story from one dataset, not a gallery of charts.

## Story structure

1. Question.
2. Context.
3. Evidence.
4. Interpretation.
5. Limitations.
6. Decision.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 30)

ROOT = Path.cwd()
LOCAL_DATA = ROOT / "data" / "snapshots"
if not LOCAL_DATA.exists():
    LOCAL_DATA = ROOT.parent / "data" / "snapshots"
print("Data path:", LOCAL_DATA.resolve())

In [ ]:
path = sorted(LOCAL_DATA.glob("*.csv"))[0]
df = pd.read_csv(path)
df = df.rename(columns={df.columns[0]: "date"})
df["date"] = pd.to_datetime(df["date"])
value_cols = [c for c in df.columns if c not in {"date", "isPartial"}]
df[value_cols] = df[value_cols].apply(pd.to_numeric, errors="coerce")
df.head()

In [ ]:
long = df.melt(id_vars="date", value_vars=value_cols, var_name="signal", value_name="index").dropna()
long["rolling_4"] = long.groupby("signal")["index"].transform(lambda s: s.rolling(4, min_periods=1).mean())
long.head()

In [ ]:
plt.figure(figsize=(12, 5))
sns.lineplot(data=long, x="date", y="rolling_4", hue="signal")
plt.title("Smoothed attention signals")
plt.xlabel("")
plt.ylabel("4-period rolling index")
plt.legend(title="Signal")
plt.show()

In [ ]:
recent = long.sort_values("date").groupby("signal").tail(8)
ranking = recent.groupby("signal")["index"].mean().sort_values(ascending=False).to_frame("recent_mean")
ranking

In [ ]:
baseline = long.groupby("signal")["index"].median()
latest = long.sort_values("date").groupby("signal").tail(1).set_index("signal")["index"]
lift = ((latest - baseline) / baseline.replace(0, np.nan)).sort_values(ascending=False).to_frame("lift_vs_median")
lift.round(2)

## Narrative cell

**Insight:** ...

**Evidence:** ...

**Decision:** ...

**Limits:** Google Trends is normalized attention, not demand, revenue or causality.